# Fifth-paper offline pilot: Colab + Google Drive
Compute runs in this Colab runtime; data, models, logs and results persist in MyDrive/research5.
Use normal Google authorization. The authorized frozen 10-person development queue follows compatible A/B gates; no new method training occurs.
Colab runtime limits still apply; the computer can close after server-side execution starts, but a runtime can be reclaimed.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
project = Path('/content/drive/MyDrive/research5')
assert project.is_dir(), 'Use the existing MyDrive/research5 project folder'
# Set this to the actual freshly uploaded bundle name.
bundle = project / 'bundles/research5-colab-drive.zip'
assert bundle.is_file(), f'Fresh code bundle not found: {bundle}'
print('Persistent project:', project)


In [ ]:
import zipfile, hashlib
bundle_hash = hashlib.sha256(bundle.read_bytes()).hexdigest()
extract_root = Path('/content/research5-upload') / bundle_hash[:20]
extract_root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(bundle) as archive:
    for item in archive.infolist():
        target = (extract_root / item.filename).resolve()
        assert target.is_relative_to(extract_root.resolve()), 'Unsafe archive path'
    archive.extractall(extract_root)
candidates = list(extract_root.glob('**/pilot/colab_drive_run.py'))
assert len(candidates) == 1, 'Bundle must contain exactly one pilot driver'
source = candidates[0].parents[1]
print('Uploaded bundle SHA256:', bundle_hash)
print('Source:', source)


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, str(source / 'pilot/colab_drive_run.py'),
    '--source-root', str(source), '--project-root', str(project), '--setup-only'], check=True)


In [ ]:
# Keep False for ordinary use. Set True only after a previous runtime stopped
# and its Drive lock/heartbeat is older than 10 minutes.
recover_stale_lock = False
# The user authorized this frozen 10-person queue. Set False to run A/B only.
development_queue = True
command = [sys.executable, str(source / 'pilot/colab_drive_run.py'),
    '--source-root', str(source), '--project-root', str(project)]
if recover_stale_lock:
    command.append('--recover-stale-lock')
if development_queue:
    command.append('--development-queue')
subprocess.run(command, check=True)


In [ ]:
import json
for state_path in sorted((project / 'status').glob('*/controller_status.json')):
    print(state_path.parent.name, json.loads(state_path.read_text()))
print('Results and logs are already saved in Drive; no end-of-session download is required.')
